In [5]:
import ollama

# Definiálunk egy "get_weather" toolt
tools = [
    {
        'name': 'get_weather',
        'description': 'Get current weather by city name',
        'parameters': {
            'type': 'object',
            'properties': {
                'city': {'type': 'string'}
            },
            'required': ['city']
        }
    }
]

# Első lekérés: a modell kérni fogja a tool meghívását
response = ollama.chat(
    model='gpt-oss:20b',
    messages=[
        {'role': 'system', 'content': 'You can use the tool get_weather to retrieve weather information.'},
        {'role': 'user', 'content': 'What is the weather in Debrecen right now?'}
    ],
    tools=tools
)

print("Model response:", response)

# Példa: ha a modell tool hívást adott vissza:
if 'tool_calls' in response['message']:
    for tool_call in response['message']['tool_calls']:
        
        if tool_call.function.name == 'get_weather':
            city = tool_call.function.arguments['location']  # nem 'city', hanem 'location' a kulcs    
            # Itt kéne valódi időjárás API-t hívni, pl. OpenWeatherMap
            weather_info = {"temp": 28, "condition": "Sunny"}

            # Második kör: visszaadjuk a tool eredményét a modellnek
            followup = ollama.chat(
                model='gpt-oss:20b',
                messages=[
                    *response['message'],
                    {
                        'role': 'tool',
                        'name': 'get_weather',
                        'content': str(weather_info)
                    }
                ]
            )
            print("Final answer:", followup)


Model response: model='gpt-oss:20b' created_at='2025-08-08T12:05:04.599613601Z' done=True done_reason='stop' total_duration=894369670 load_duration=57691088 prompt_eval_count=132 prompt_eval_duration=34375665 eval_count=34 eval_duration=797144534 message=Message(role='assistant', content='', thinking='We need to use get_weather function.', images=None, tool_calls=[ToolCall(function=Function(name='get_weather', arguments={'location': 'Debrecen'}))])


KeyError: 'messages'